# Search Strategy Refinement, Method Comparison

*Author: Regina Chua*

> This notebook tests three complementary methods for refining the search strategy and pre-ranking
> the corpus against a small set of manually confirmed seed papers. The goal is to compare which
> method surfaces the most relevant articles and what new vocabulary it suggests, feeding back into
> `search_strategy.py` before the full multi-database run.

**Methods tested:**

| Method | What it does | Best for |
|---|---|---|
| **TF-IDF** | Term weighting + cosine similarity to seed query | Baseline ranking; corpus-level keyword surfacing |
| **BM25** | Term-frequency ranking with length normalisation | Keyword recall, interpretable scores |
| **KeyBERT / YAKE** | Keyphrase extraction from seed abstracts | Surfacing new vocabulary for query expansion |
| **SPECTER embeddings** | Semantic similarity in dense vector space | Finding papers that use different terminology |

**Workflow:** populate `seed_papers.csv` → run top-to-bottom → inspect rankings and candidate
keyphrases → update `search_strategy.py` with any new terms.

---

### Seed set guidance

> For reliable ranking:
>
> | Size | What it enables |
> |---|---|
> | **≥ 15** | Minimum for stable BM25 and embedding centroids |
> | **20–30** | Recommended — covers enough sub-theme diversity |
> | **≥ 50** | Needed for the calibration step (precision/recall tuning) |

## 1. Environment Setup

In [ ]:
%pip install --upgrade pip
%pip install rank-bm25 sentence-transformers keybert yake pymed --quiet

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [28]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

SEED_PATH   = Path("seed_papers.csv")
# Clean UTF-8 export of the same seed papers — run build_refinement_corpus.py after seed updates.
CORPUS_PATH = Path("refinement_corpus.csv")
TOP_N       = 20   # how many top-ranked corpus papers to show per method


def _normalize_term(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()


def _criteria_term_set(criteria_dict):
    terms = set()
    for group in criteria_dict.values():
        for t in group:
            terms.add(_normalize_term(t.replace("*", "")))
    return terms


def _strategy_term_set():
    """Normalized terms in the full live strategy (for BM25 / KeyBERT dedup)."""
    from search_strategy import ALTERNATE_TERMS, EXCLUSION_TERMS, INCLUSION_CRITERIA

    terms = _criteria_term_set(INCLUSION_CRITERIA) | _criteria_term_set(ALTERNATE_TERMS)
    for t in EXCLUSION_TERMS:
        terms.add(_normalize_term(t.replace("*", "")))
    return terms


def _initial_criteria_term_set():
    """Normalized terms in INITIAL_CRITERIA (for TF-IDF expansion baseline)."""
    from search_strategy import INITIAL_CRITERIA

    return _criteria_term_set(INITIAL_CRITERIA)


def _matches_initial(term_norm, initial_terms):
    if term_norm in initial_terms:
        return True
    return any(
        len(init) > 3 and (init in term_norm or term_norm in init)
        for init in initial_terms
    )


def _build_category_anchors():
    """Token anchors per category for assigning new terms to disease / spatial / exposure."""
    from search_strategy import ALTERNATE_TERMS, INCLUSION_CRITERIA, INITIAL_CRITERIA

    anchors = {cat: set() for cat in ("disease", "spatial", "exposure")}
    for criteria in (INITIAL_CRITERIA, INCLUSION_CRITERIA, ALTERNATE_TERMS):
        for cat, terms in criteria.items():
            for t in terms:
                norm = _normalize_term(t.replace("*", ""))
                anchors[cat].add(norm)
                anchors[cat].update(norm.split())
    return anchors


def _classify_term(term_norm, category_anchors):
    term_tokens = set(term_norm.split())
    best_cat, best_score = "exposure", -1
    for cat, anchor in category_anchors.items():
        score = len(term_tokens & anchor) + sum(
            1 for a in anchor if len(a) > 3 and a in term_norm
        )
        if score > best_score:
            best_cat, best_score = cat, score
    return best_cat


existing_strategy_terms = _strategy_term_set()
initial_criteria_terms = _initial_criteria_term_set()
category_anchors = _build_category_anchors()

print("Environment ready.")

Environment ready.


## 2. Load Seed Papers & Corpus

> The seed set is loaded from `seed_papers.csv` (55 confirmed-relevant papers). The **corpus**
> is the same papers, cleaned and exported to `refinement_corpus.csv` by
> `build_refinement_corpus.py` (encoding fixes, title + abstract required). Re-run that
> script whenever `seed_papers.csv` changes.

In [3]:
def _combine_text(row):
    """Join title, abstract, and keywords into one string for indexing."""
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("abstract", "") or ""),
        str(row.get("keywords", "") or ""),
    ]
    return " ".join(p for p in parts if p).strip()


def _tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()


# --- Seed papers ---
if not SEED_PATH.exists():
    raise FileNotFoundError(
        f"'{SEED_PATH}' not found. Add your confirmed-relevant papers to that file "
        "(title + abstract required) and re-run this cell."
    )

df_seed = pd.read_csv(SEED_PATH, encoding="latin-1")
# Drop the placeholder row if it hasn't been replaced yet
df_seed = df_seed[~df_seed["title"].astype(str).str.startswith("REPLACE")].reset_index(drop=True)

if len(df_seed) == 0:
    raise ValueError(
        "seed_papers.csv contains no real entries yet. Fill in at least one confirmed-relevant "
        "paper (title + abstract) and re-run."
    )

df_seed["_text"] = df_seed.apply(_combine_text, axis=1)
seed_texts = df_seed["_text"].tolist()
print(f"Seed papers loaded: {len(df_seed)}")
if len(df_seed) < 15:
    print(f"  ⚠  {len(df_seed)} papers is below the recommended minimum of 15. "
          "Rankings will be less reliable — add more confirmed-relevant papers.")
display(df_seed[[c for c in ["title","doi","pubmed_id"] if c in df_seed.columns]].head())

# --- Corpus ---
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"'{CORPUS_PATH}' not found. Run build_refinement_corpus.py first:\n"
        "  python build_refinement_corpus.py"
    )

df_corpus = pd.read_csv(CORPUS_PATH, encoding="utf-8")
df_corpus["_text"] = df_corpus.apply(_combine_text, axis=1)
# Drop rows with no usable text
df_corpus = df_corpus[df_corpus["_text"].str.strip().astype(bool)].reset_index(drop=True)
print(f"\nCorpus loaded: {len(df_corpus)} documents from '{CORPUS_PATH.name}'")

Seed papers loaded: 55


,title,doi,pubmed_id
0,Machine Learning Models for Predicting Parkinsonâs Disease Progression Using Longitudinal Data...,10.9734/ajrcos/2025/v18i3593,NaN
1,Variations in the patterns of prevalence and therapy in Australasian Parkinsonâs disease patie...,10.1136/bmjno-2019-000033,NaN
2,Literature review and meta-analysis of environmental toxins associated with increased risk of Pa...,10.1016/j.scitotenv.2024.172838,NaN
3,The epidemiology of Parkinson's disease,10.1016/s0140-6736(23)01419-8,NaN
4,Hierarchical Bayesian modeling of spatio-temporal area-interaction processes,10.1016/j.csda.2021.107349,NaN



Corpus loaded: 55 documents from 'refinement_corpus.csv'


## 3. Method 1 — TF-IDF

> Baseline method (same family as `pubmed.ipynb`). **Article ranking:** cosine similarity
> between each corpus document and the concatenated seed text in a shared TF-IDF space.
> **Keywords:** corpus-level 1–3 gram terms ranked by TF-IDF + frequency, compared against
> `INITIAL_CRITERIA` in `search_strategy.py`. New terms are assigned to disease / spatial /
> exposure and saved in `df_tfidf_expanded` for comparison with later methods.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

NGRAM_RANGE = (1, 3)
MAX_DF = 0.95

# --- Articles: query–document cosine similarity ---
doc_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_df=MAX_DF)
X_corpus = doc_vectorizer.fit_transform(df_corpus["_text"])
X_query = doc_vectorizer.transform([" ".join(seed_texts)])

df_tfidf = df_corpus.copy()
df_tfidf["tfidf_score"] = cosine_similarity(X_query, X_corpus)[0]
df_tfidf = df_tfidf.sort_values("tfidf_score", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} TF-IDF-ranked articles (cosine similarity to seed query):")
show_cols = [c for c in ["title", "tfidf_score", "doi", "publication_date"] if c in df_tfidf.columns]
display(df_tfidf[show_cols].head(TOP_N).style.format({"tfidf_score": "{:.3f}"}))

# --- Keywords: corpus-level n-gram ranking ---
kw_vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, stop_words="english", max_df=MAX_DF)
X_kw = kw_vectorizer.fit_transform(df_corpus["_text"])
kw_tfidf = np.asarray(X_kw.sum(axis=0)).ravel()
kw_names = kw_vectorizer.get_feature_names_out()

count_vec = CountVectorizer(ngram_range=NGRAM_RANGE, stop_words="english")
Y = count_vec.fit_transform(df_corpus["_text"])
kw_counts = np.asarray(Y.sum(axis=0)).ravel()

tfidf_keywords = (
    pd.DataFrame({"term": kw_names, "tfidf": kw_tfidf})
    .merge(
        pd.DataFrame({"term": count_vec.get_feature_names_out(), "count": kw_counts}),
        on="term",
        how="outer",
    )
    .fillna(0)
)
tfidf_keywords["ngram_len"] = tfidf_keywords["term"].str.count(" ") + 1
tfidf_keywords["score"] = (
    tfidf_keywords["tfidf"] * 0.7
    + (tfidf_keywords["count"] / (tfidf_keywords["count"].max() + 1e-9)) * 0.3
)
tfidf_keywords["term_norm"] = tfidf_keywords["term"].apply(_normalize_term)
tfidf_keywords["already_in_initial"] = tfidf_keywords["term_norm"].apply(
    lambda t: _matches_initial(t, initial_criteria_terms)
)
tfidf_keywords = tfidf_keywords[
    (tfidf_keywords["term"].str.len() > 2) & (~tfidf_keywords["term"].str.match(r"^\d+$"))
]
tfidf_keywords = tfidf_keywords.sort_values("score", ascending=False).reset_index(drop=True)

print(f"\nTop {TOP_N} corpus keywords (TF-IDF + frequency):")
display(
    tfidf_keywords[["term", "ngram_len", "count", "tfidf", "score", "already_in_initial"]]
    .head(TOP_N)
    .style.format({"tfidf": "{:.2f}", "score": "{:.2f}"})
)

# Expanded terms by category (new vs INITIAL_CRITERIA) — for cross-method comparison
new_tfidf = tfidf_keywords[~tfidf_keywords["already_in_initial"]].copy()
new_tfidf["category"] = new_tfidf["term_norm"].apply(
    lambda t: _classify_term(t, category_anchors)
)

expansions_by_cat = {
    cat: (
        new_tfidf[new_tfidf["category"] == cat]
        .sort_values("score", ascending=False)["term"]
        .head(TOP_N)
        .tolist()
    )
    for cat in ("disease", "spatial", "exposure")
}
max_rows = max((len(v) for v in expansions_by_cat.values()), default=1)

df_tfidf_expanded = pd.DataFrame({
    cat: expansions_by_cat[cat] + [""] * (max_rows - len(expansions_by_cat[cat]))
    for cat in ("disease", "spatial", "exposure")
})

df_tfidf_expansions = (
    new_tfidf.sort_values("score", ascending=False)
    .groupby("category", sort=False)
    .head(TOP_N)
    .reset_index(drop=True)[["category", "term", "score", "tfidf", "count"]]
)
df_tfidf_expansions.insert(0, "method", "tfidf")

print(f"\nTF-IDF expanded terms by category (top {TOP_N} new vs INITIAL_CRITERIA):")
display(df_tfidf_expanded.replace("", pd.NA))

Top 20 TF-IDF-ranked articles (cosine similarity to seed query):


,title,tfidf_score,doi,publication_date
0,Association of NO2 and Other Air Pollution Exposures With the Risk of Parkinson Disease,0.340,10.1001/jamaneurol.2021.1335,2021
1,Air pollution and Parkinson's disease: A prospective cohort study with sex-stratified analysis in the UK biobank,0.335,10.1016/j.neuro.2025.103353,2025-12
2,Fine Particulate Matter and Parkinson Disease Risk Among Medicare Beneficiaries,0.334,10.1212/wnl.0000000000207871,21-11-23
3,Associations between long-term exposure to ambient air pollution and Parkinson's disease prevalence: A cross-sectional study,0.332,10.1016/j.neuint.2019.104615,2020-2
4,Long-term air pollution exposure and Parkinson's disease mortality in a large pooled European cohort: An ELAPSE study,0.329,10.1016/j.envint.2022.107667,2023-1
5,Pesticide Exposure and Parkinson's Disease: A Qualitative Study of Experiences in Ireland,0.327,10.1111/hex.70329,2025-10
6,Occupational pesticide exposure and the risk of death in patients with Parkinson's disease: an observational study in southern Brazil,0.316,10.1186/s12940-020-00624-8,2020-12
7,"Air pollution, surrounding green, road proximity and Parkinson's disease: A prospective cohort study",0.311,10.1016/j.envres.2021.111170,2021-6
8,Well Water and Parkinson's Disease in Medicare Beneficiaries: A Nationwide Case-Control Study,0.304,10.3233/jpd-191793,03-04-20
9,Pesticide use and incident Parkinson's disease in a cohort of farmers and their spouses,0.296,10.1016/j.envres.2020.110186,2020-12



Top 20 corpus keywords (TF-IDF + frequency):


,term,ngram_len,count,tfidf,score,already_in_initial
0,disease,1,170,1.76,1.45,True
1,parkinson,1,149,1.70,1.39,True
2,parkinson disease,2,126,1.44,1.18,True
3,exposure,1,99,1.49,1.18,False
4,risk,1,89,1.38,1.08,False
5,study,1,85,1.07,0.86,False
6,pm2,1,45,1.14,0.86,False
7,pesticides,1,49,1.12,0.85,False
8,pesticide,1,43,1.08,0.81,False
9,prevalence,1,39,0.97,0.73,False



TF-IDF expanded terms by category (top 20 new vs INITIAL_CRITERIA):


,disease,spatial,exposure
0,risk,analysis,exposure
1,study,information,pesticides
2,pm2,atmospheric,pesticide
3,prevalence,epidemiology,pollution
4,health,atmospheric risk,air
5,age,atmospheric risk factors,air pollution
6,use,atmospheric factor,pesticide exposure
7,data,meta analysis,water
8,model,analysis using,exposures
9,people,atmospheric factors,occupational


In [5]:
from search_strategy import INITIAL_CRITERIA


def _print_list(title, items):
    print(title)
    if not items:
        print("  (none)")
    else:
        for item in items:
            print(f"  - {item}")
    print()


# 1. All terms in INITIAL_CRITERIA
initial_terms = sorted(
    {t for group in INITIAL_CRITERIA.values() for t in group},
    key=str.lower,
)

# 2. TF-IDF keywords that overlap INITIAL_CRITERIA
tfidf_matched = tfidf_keywords[tfidf_keywords["already_in_initial"]]["term"].tolist()

# 3. Top 20 TF-IDF keywords not in INITIAL_CRITERIA
tfidf_new = (
    tfidf_keywords[~tfidf_keywords["already_in_initial"]]
    .head(TOP_N)["term"]
    .tolist()
)

_print_list(f"1. In INITIAL_CRITERIA ({len(initial_terms)} terms):", initial_terms)
_print_list(f"2. TF-IDF matched to INITIAL_CRITERIA ({len(tfidf_matched)} terms):", tfidf_matched)
_print_list(f"3. TF-IDF new — top {TOP_N} not in INITIAL_CRITERIA:", tfidf_new)

1. In INITIAL_CRITERIA (4 terms):
  - environment
  - geographic
  - geospatial
  - parkinson* disease

2. TF-IDF matched to INITIAL_CRITERIA (524 terms):
  - disease
  - parkinson
  - parkinson disease
  - environmental
  - geographic
  - parkinson disease pd
  - iron
  - people parkinson disease
  - geospatial
  - environmental atmospheric
  - environmental factors
  - environment
  - spatial
  - geospatial analysis
  - men
  - risk parkinson disease
  - pollution parkinson disease
  - geographic demographic
  - geographic demographic profile
  - environmental exposures
  - pink1 parkinson disease
  - environmental atmospheric risk
  - geographical
  - environments
  - onset parkinson disease
  - analysis environmental
  - incidence parkinson disease
  - parkinson disease related
  - parkinson disease patients
  - parkinson disease brain
  - environmental goods
  - analysis environmental atmospheric
  - geographical analysis
  - disease environmental atmospheric
  - geospatial analys

## 4. Method 2 — BM25

> BM25 extends TF-IDF with document-length normalisation and term saturation. **Article
> ranking:** score each corpus document against the concatenated seed query. **Keywords:**
> seed-query tokens weighted by BM25 IDF (rare in the corpus × frequent in the query).
> Compare rankings and keyword lists with Section 3 to see what BM25 adds.

In [6]:
from collections import Counter

from rank_bm25 import BM25Okapi

# --- Articles ---
corpus_tokens = [_tokenize(t) for t in df_corpus["_text"]]
bm25 = BM25Okapi(corpus_tokens)

seed_query_tokens = _tokenize(" ".join(seed_texts))
bm25_scores = bm25.get_scores(seed_query_tokens)

df_bm25 = df_corpus.copy()
df_bm25["bm25_score"] = bm25_scores
df_bm25 = df_bm25.sort_values("bm25_score", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} BM25-ranked articles:")
show_cols = [c for c in ["title", "bm25_score", "doi", "publication_date"] if c in df_bm25.columns]
display(df_bm25[show_cols].head(TOP_N).style.format({"bm25_score": "{:.2f}"}))

# --- Keywords: query tokens weighted by BM25 IDF ---
query_counts = Counter(seed_query_tokens)
bm25_keywords = (
    pd.DataFrame([
        {"term": term, "query_freq": query_counts[term], "idf": bm25.idf.get(term, 0.0)}
        for term in query_counts
        if term in bm25.idf and len(term) > 2
    ])
    .assign(bm25_weight=lambda d: d["query_freq"] * d["idf"])
    .sort_values("bm25_weight", ascending=False)
    .reset_index(drop=True)
)
bm25_keywords["term_norm"] = bm25_keywords["term"].apply(_normalize_term)
bm25_keywords["already_in_strategy"] = bm25_keywords["term_norm"].isin(existing_strategy_terms)

print(f"\nTop {TOP_N} query keywords (BM25 IDF x seed query frequency):")
display(
    bm25_keywords[["term", "query_freq", "idf", "bm25_weight", "already_in_strategy"]]
    .head(TOP_N)
    .style.format({"idf": "{:.2f}", "bm25_weight": "{:.2f}"})
)

Top 20 BM25-ranked articles:


,title,bm25_score,doi,publication_date
0,"Air pollution, surrounding green, road proximity and Parkinson's disease: A prospective cohort study",9395.12,10.1016/j.envres.2021.111170,2021-6
1,Associations between long-term exposure to ambient air pollution and Parkinson's disease prevalence: A cross-sectional study,9354.45,10.1016/j.neuint.2019.104615,2020-2
2,Pesticide Exposure and Parkinson's Disease: A Qualitative Study of Experiences in Ireland,9334.73,10.1111/hex.70329,2025-10
3,Association of NO2 and Other Air Pollution Exposures With the Risk of Parkinson Disease,9289.12,10.1001/jamaneurol.2021.1335,2021
4,Long-term air pollution exposure and Parkinson's disease mortality in a large pooled European cohort: An ELAPSE study,9256.15,10.1016/j.envint.2022.107667,2023-1
5,"Global burden of 369 diseases and injuries in 204 countries and territories, 1990–2019: a systematic analysis for the Global Burden of Disease Study 2019",9069.17,10.1016/s0140-6736(20)30925-9,2020-10
6,Occupational pesticide exposure and the risk of death in patients with Parkinson's disease: an observational study in southern Brazil,9044.96,10.1186/s12940-020-00624-8,2020-12
7,Barriers and Facilitators to Accessing Healthcare for People With Parkinson's Disease in Latin America: A Qualitative Study,8923.22,10.1111/hex.70380,2025-8
8,Literature review and meta-analysis of environmental toxins associated with increased risk of Parkinson's disease,8900.58,10.1016/j.scitotenv.2024.172838,2024-6
9,Proximity to residential and workplace pesticides application and the risk of progression of Parkinson's diseases in Central California,8887.30,10.1016/j.scitotenv.2022.160851,2023-3



Top 20 query keywords (BM25 IDF x seed query frequency):


,term,query_freq,idf,bm25_weight,already_in_strategy
0,and,575,0.77,443.06,False
1,the,569,0.77,438.43,False
2,with,208,0.77,160.27,False
3,disease,170,0.77,130.99,False
4,parkinson,149,0.77,114.81,False
5,for,149,0.77,114.81,False
6,were,114,0.77,87.84,False
7,pm2,45,1.87,84.00,False
8,exposure,99,0.77,76.28,False
9,was,96,0.77,73.97,False


## 5. Method 3 — Keyphrase Extraction (KeyBERT + YAKE)

> Both methods extract keyphrases from the seed abstracts, but use different signals. **YAKE** uses
> statistical co-occurrence (fast, no model download). **KeyBERT** uses contextual embeddings (richer,
> slower). The union of their output is a candidate list of terms to add to `search_strategy.py`
> — specifically to `ALTERNATE_TERMS` or `INCLUSION_CRITERIA`. I mark any term that already exists
> in the strategy so it's easy to spot the genuinely new ones.

In [7]:
import yake
from keybert import KeyBERT

from search_strategy import INCLUSION_CRITERIA, ALTERNATE_TERMS

# All terms already in the strategy (for deduplication display)
existing_terms = set()
for group in list(INCLUSION_CRITERIA.values()) + list(ALTERNATE_TERMS.values()):
    for t in group:
        existing_terms.add(t.lower().replace("*", ""))

seed_corpus_text = " ".join(seed_texts)

# --- YAKE ---
yake_extractor = yake.KeywordExtractor(
    lan="en", n=3, dedupLim=0.7, top=30, features=None
)
yake_kws = yake_extractor.extract_keywords(seed_corpus_text)
# YAKE scores are inverted (lower = more important)
yake_df = pd.DataFrame(yake_kws, columns=["keyphrase", "yake_score"]).sort_values("yake_score")
yake_df["already_in_strategy"] = yake_df["keyphrase"].str.lower().isin(existing_terms)

print("--- YAKE keyphrases (lower score = more relevant) ---")
display(yake_df.head(20).style.format({"yake_score": "{:.4f}"}))

# --- KeyBERT ---
# Uses a lightweight all-MiniLM model by default (fast). Swap for 'allenai-specter'
# if you want scientific-domain embeddings (requires the SPECTER model to be downloaded first).
print("\nLoading KeyBERT model (this may take a moment on first run)...")
kw_model = KeyBERT()
keybert_kws = kw_model.extract_keywords(
    seed_corpus_text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    top_n=30,
    diversity=0.6,   # MMR diversity — avoids near-duplicate keyphrases
)
keybert_df = pd.DataFrame(keybert_kws, columns=["keyphrase", "keybert_score"]).sort_values(
    "keybert_score", ascending=False
)
keybert_df["already_in_strategy"] = keybert_df["keyphrase"].str.lower().isin(existing_terms)

print("\n--- KeyBERT keyphrases (higher score = more relevant) ---")
display(keybert_df.head(20).style.format({"keybert_score": "{:.3f}"}))

# --- Union of new terms ---
new_yake = set(yake_df[~yake_df["already_in_strategy"]]["keyphrase"].str.lower())
new_keybert = set(keybert_df[~keybert_df["already_in_strategy"]]["keyphrase"].str.lower())
new_terms_union = sorted(new_yake | new_keybert)
print(f"\n{len(new_terms_union)} candidate new terms (not already in search_strategy.py):")
for t in new_terms_union:
    print(f"  {t}")

--- YAKE keyphrases (lower score = more relevant) ---


,keyphrase,yake_score,already_in_strategy
0,Parkinson disease,0.0001,True
1,Parkinson disease Parkinson,0.0004,False
2,Parkinson Disease Risk,0.0005,False
3,Parkinson,0.0006,False
4,Disease,0.0006,False
5,disease Parkinson disease,0.0006,False
6,Parkinson disease prevalence,0.0008,False
7,Parkinson Disease Brain,0.0012,False
8,Parkinson Disease Background,0.0013,False
9,Background Parkinson disease,0.0013,False



Loading KeyBERT model (this may take a moment on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8373.00it/s]



--- KeyBERT keyphrases (higher score = more relevant) ---


,keyphrase,keybert_score,already_in_strategy
0,models predicting parkinsonâ,0.686,False
1,predicting parkinsonâ,0.661,False
2,predicting parkinsonâ disease,0.661,False
3,parkinson disease data,0.634,False
4,parkinson disease cohort,0.625,False
5,parkinson limited multidisciplinary,0.616,False
6,parkinson results 2013,0.608,False
7,parkinson disease prospective,0.600,False
8,research parkinson,0.600,False
9,progression parkinson,0.595,False



56 candidate new terms (not already in search_strategy.py):
  abstract background parkinson
  assessment additional parkinson
  background parkinson disease
  data certain parkinson
  disease
  disease epidemiology parkinson
  disease parkinson disease
  disease risk
  diseases
  epidemiology parkinson
  epidemiology parkinson disease
  exposure
  human parkinson disease
  introduction parkinson disease
  methods information parkinson
  models predicting parkinsonâ
  nan parkinson disease
  occupational pesticide exposure
  onset parkinson disease
  parkinson
  parkinson disease background
  parkinson disease brain
  parkinson disease cases
  parkinson disease cohort
  parkinson disease community
  parkinson disease compared
  parkinson disease data
  parkinson disease elucidating
  parkinson disease epidemiology
  parkinson disease importance
  parkinson disease parkinson
  parkinson disease prevalence
  parkinson disease prospective
  parkinson disease results
  parkinson disease ri

## 6. Method 4 — SPECTER Semantic Similarity

> SPECTER is a transformer model trained specifically for scientific document similarity. I embed
> each seed paper and each corpus document, then score each corpus document by its **cosine
> similarity to the seed centroid** (the average of the seed embeddings). This catches papers that
> are conceptually similar to the seed set but use different terminology — the gap that keyword
> methods leave open.
>
> First run downloads the SPECTER model (~400 MB); subsequent runs use the cached version.
> If the download is too slow, replace `'allenai-specter'` with `'all-MiniLM-L6-v2'` for a
> faster ~80 MB model (slightly less domain-specific).

In [8]:
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "allenai-specter"   # swap for "all-MiniLM-L6-v2" if you want a faster, smaller model

print(f"Loading {MODEL_NAME} (downloads ~400 MB on first run, then cached) ...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")

# Embed seed papers
print("Embedding seed papers ...")
seed_embeddings = model.encode(seed_texts, show_progress_bar=True, convert_to_tensor=True)
seed_centroid = seed_embeddings.mean(dim=0)  # single representative vector

# Embed corpus
print("Embedding corpus (may take a few minutes) ...")
corpus_embeddings = model.encode(
    df_corpus["_text"].tolist(), show_progress_bar=True, convert_to_tensor=True
)

# Score each corpus document against the seed centroid
sim_scores = util.cos_sim(seed_centroid.unsqueeze(0), corpus_embeddings)[0].cpu().numpy()

df_specter = df_corpus.copy()
df_specter["specter_sim"] = sim_scores
df_specter = df_specter.sort_values("specter_sim", ascending=False).reset_index(drop=True)

print(f"\nTop {TOP_N} SPECTER-ranked corpus documents:")
show_cols = [c for c in ["title", "specter_sim", "doi", "publication_date"] if c in df_specter.columns]
display(df_specter[show_cols].head(TOP_N).style.format({"specter_sim": "{:.3f}"}))

Loading allenai-specter (downloads ~400 MB on first run, then cached) ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8474.03it/s]


Model loaded.
Embedding seed papers ...


Batches: 100%|██████████| 2/2 [00:07<00:00,  3.79s/it]


Embedding corpus (may take a few minutes) ...


Batches: 100%|██████████| 2/2 [00:07<00:00,  3.61s/it]


Top 20 SPECTER-ranked corpus documents:


,title,specter_sim,doi,publication_date
0,Fine Particulate Matter and Parkinson Disease Risk Among Medicare Beneficiaries,0.935,10.1212/wnl.0000000000207871,21-11-23
1,Literature review and meta-analysis of environmental toxins associated with increased risk of Parkinson's disease,0.931,10.1016/j.scitotenv.2024.172838,2024-6
2,Air pollution and Parkinson's disease: A prospective cohort study with sex-stratified analysis in the UK biobank,0.931,10.1016/j.neuro.2025.103353,2025-12
3,Well Water and Parkinson's Disease in Medicare Beneficiaries: A Nationwide Case-Control Study,0.929,10.3233/jpd-191793,03-04-20
4,Geographic and Ethnic Variation in Parkinson Disease: A Population-Based Study of US Medicare Beneficiaries,0.924,10.1159/000275491,2010
5,Associations between long-term exposure to ambient air pollution and Parkinson's disease prevalence: A cross-sectional study,0.924,10.1016/j.neuint.2019.104615,2020-2
6,The epidemiology of Parkinson's disease,0.922,10.1016/s0140-6736(23)01419-8,2024-1
7,Occupational pesticide exposure and the risk of death in patients with Parkinson's disease: an observational study in southern Brazil,0.920,10.1186/s12940-020-00624-8,2020-12
8,Long-term air pollution exposure and Parkinson's disease mortality in a large pooled European cohort: An ELAPSE study,0.919,10.1016/j.envint.2022.107667,2023-1
9,Proximity to residential and workplace pesticides application and the risk of progression of Parkinson's diseases in Central California,0.917,10.1016/j.scitotenv.2022.160851,2023-3


## 7. Compare Rankings

> Merges the BM25 and SPECTER rankings into one table so it's easy to spot:
> - Papers that rank high on **both** — very likely relevant.
> - Papers high on SPECTER but low on BM25 — semantically similar but may use different vocabulary;
>   their titles/abstracts are good candidates for new search terms.
> - Papers high on BM25 but low on SPECTER — keyword-rich but may be thematically tangential;
>   worth a quick scan to check whether they are noise or whether the seed set is missing a sub-theme.

In [9]:
# Normalise both scores to [0, 1] for a fair side-by-side comparison
def _norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0


bm25_rank = df_bm25[["title", "doi", "bm25_score"]].copy()
bm25_rank["bm25_norm"] = _norm(bm25_rank["bm25_score"])
bm25_rank["bm25_rank"] = bm25_rank["bm25_score"].rank(ascending=False).astype(int)

specter_rank = df_specter[["title", "doi", "specter_sim"]].copy()
specter_rank["specter_norm"] = _norm(specter_rank["specter_sim"])
specter_rank["specter_rank"] = specter_rank["specter_sim"].rank(ascending=False).astype(int)

merged = bm25_rank.merge(specter_rank[["doi", "specter_norm", "specter_rank"]], on="doi", how="inner")
merged["combined_norm"] = (merged["bm25_norm"] + merged["specter_norm"]) / 2
merged = merged.sort_values("combined_norm", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} papers by combined BM25 + SPECTER score:")
display(
    merged[["title", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]]
    .head(TOP_N)
    .style.format({
        "bm25_norm":     "{:.3f}",
        "specter_norm":  "{:.3f}",
        "combined_norm": "{:.3f}",
    })
    .background_gradient(subset=["combined_norm"], cmap="YlGn")
)

# Flag interesting divergences
divergent = merged[abs(merged["bm25_rank"] - merged["specter_rank"]) > 50].head(10)
if not divergent.empty:
    print(f"\nPapers with large rank divergence (>50 positions) — worth investigating:")
    display(divergent[["title", "bm25_rank", "specter_rank"]].reset_index(drop=True))

Top 20 papers by combined BM25 + SPECTER score:


,title,bm25_rank,specter_rank,bm25_norm,specter_norm,combined_norm
0,Associations between long-term exposure to ambient air pollution and Parkinson's disease prevalence: A cross-sectional study,2,6,0.989,0.954,0.972
1,Long-term air pollution exposure and Parkinson's disease mortality in a large pooled European cohort: An ELAPSE study,5,9,0.964,0.932,0.948
2,"Air pollution, surrounding green, road proximity and Parkinson's disease: A prospective cohort study",1,15,1.000,0.890,0.945
3,Association of NO2 and Other Air Pollution Exposures With the Risk of Parkinson Disease,4,12,0.973,0.917,0.945
4,Fine Particulate Matter and Parkinson Disease Risk Among Medicare Beneficiaries,13,1,0.859,1.000,0.930
5,Literature review and meta-analysis of environmental toxins associated with increased risk of Parkinson's disease,9,2,0.872,0.985,0.929
6,Occupational pesticide exposure and the risk of death in patients with Parkinson's disease: an observational study in southern Brazil,7,8,0.909,0.940,0.925
7,Well Water and Parkinson's Disease in Medicare Beneficiaries: A Nationwide Case-Control Study,14,4,0.849,0.977,0.913
8,Pesticide Exposure and Parkinson's Disease: A Qualitative Study of Experiences in Ireland,3,19,0.984,0.827,0.906
9,Air pollution and Parkinson's disease: A prospective cohort study with sex-stratified analysis in the UK biobank,17,3,0.814,0.983,0.899


## 8. Export Ranked Results

> Save all three score columns to a CSV so the rankings can be reviewed outside the notebook and
> used as a reference when updating `search_strategy.py`.

In [10]:
from pathlib import Path

# Attach scores back to the full corpus metadata
full_ranked = df_corpus.drop(columns=["_text"]).merge(
    merged[["doi", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]],
    on="doi",
    how="left",
)
full_ranked = full_ranked.sort_values("combined_norm", ascending=False).reset_index(drop=True)

output_path = Path("query_refinement_ranked.csv")
full_ranked.to_csv(output_path, index=False)
print(f"Exported {len(full_ranked)} ranked records to {output_path.resolve()}")
print()
print("Next steps:")
print("  1. Review new keywords from Sections 3–5 (TF-IDF, BM25, KeyBERT/YAKE) and add useful")
print("     ones to ALTERNATE_TERMS (or INCLUSION_CRITERIA) in search_strategy.py.")
print("  2. Skim the top-ranked papers from Section 7 — check whether any sub-themes are")
print("     under-represented in the seed set.")
print("  3. If enough papers diverge between BM25 and SPECTER, consider adding a sub-theme")
print("     to the seed set to cover that vocabulary gap.")
print("  4. Once you have ≥50 labelled papers (include + confirmed exclude), run the")
print("     calibration step: score a held-out set and measure precision/recall to tune")
print("     the similarity thresholds before running LLM screening in prescreen.ipynb.")

Exported 55 ranked records to C:\Users\rchua\sysrev\query_refinement_ranked.csv

Next steps:
  1. Review new keywords from Sections 3–5 (TF-IDF, BM25, KeyBERT/YAKE) and add useful
     ones to ALTERNATE_TERMS (or INCLUSION_CRITERIA) in search_strategy.py.
  2. Skim the top-ranked papers from Section 7 — check whether any sub-themes are
     under-represented in the seed set.
  3. If enough papers diverge between BM25 and SPECTER, consider adding a sub-theme
     to the seed set to cover that vocabulary gap.
  4. Once you have ≥50 labelled papers (include + confirmed exclude), run the
     calibration step: score a held-out set and measure precision/recall to tune
     the similarity thresholds before running LLM screening in prescreen.ipynb.


## Straight from PubMed with new search strategy
Adding the abstract to build new corpus for LLM comparison and exclusion criteria building <br>
RC 21 June 2026

In [9]:
import os
from pymed import PubMed
import time
from dotenv import load_dotenv
import pandas as pd

In [10]:
# Setup PyMed instance and configuration
load_dotenv()
pubmed = PubMed(tool="SysRevSearch", email=os.getenv("PUBMED_EMAIL"))
pd.set_option("display.max_colwidth", 120)

# Load the new search results
new_results = pd.read_csv("C:/Users/rchua/sysrev/csv-parkinsonT-set.csv")
print(f"Loaded {len(new_results)} papers from new search strategy")

Loaded 255 papers from new search strategy


In [13]:
# Define text combination helper (same as earlier in notebook)
def _combine_text(row):
    """Join title, abstract, and keywords into one string for indexing."""
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("abstract", "") or ""),
        str(row.get("keywords", "") or ""),
    ]
    return " ".join(p for p in parts if p).strip()


# Detect PMID column
possible_pmid_cols = ["pubmed_id", "PMID", "pmid", "PubmedID"]
pmid_col = None

for col in possible_pmid_cols:
    if col in new_results.columns:
        pmid_col = col
        print(f"Found PMID column: '{pmid_col}'")
        break

if not pmid_col:
    # Fallback: search for any column with "pmid" or "id" in name
    pmid_cols = [c for c in new_results.columns if "pmid" in c.lower()]
    if not pmid_cols:
        pmid_cols = [c for c in new_results.columns if c.lower() == "id"]
    if pmid_cols:
        pmid_col = pmid_cols[0]
        print(f"Found PMID column: '{pmid_col}'")
    else:
        print("Available columns:")
        for c in new_results.columns:
            print(f"  - {c}")
        raise ValueError("No PMID column detected. Please check column names above.")

Found PMID column: 'PMID'


In [11]:
# Fetch abstracts from PubMed using PMID
def _fetch_abstract(pmid):
    """Fetch abstract from PubMed using PMID."""
    try:
        results = list(pubmed.query(f"{pmid}[PMID]", max_results=1))
        if results:
            return results[0].abstract if results[0].abstract else ""
        return ""
    except Exception as e:
        print(f"  Error fetching PMID {pmid}: {e}")
        return ""
    finally:
        time.sleep(0.3)  # Be respectful to NCBI servers

print(f"\nFetching abstracts from PubMed (this may take a few minutes)...")
new_results["abstract"] = new_results[pmid_col].apply(_fetch_abstract)

# Consolidate into pubmed_papers.csv
pubmed_papers = new_results.copy()
pubmed_papers.to_csv("pubmed_papers.csv", index=False, encoding="utf-8")
print(f"Saved {len(pubmed_papers)} papers to pubmed_papers.csv")

# Build corpus text for new papers
pubmed_papers["_text"] = pubmed_papers.apply(_combine_text, axis=1)
pubmed_papers = pubmed_papers[pubmed_papers["_text"].str.strip().astype(bool)].reset_index(drop=True)
print(f"Corpus ready: {len(pubmed_papers)} papers with usable text")

Found PMID column: 'PMID'

Fetching abstracts from PubMed (this may take a few minutes)...
Saved 255 papers to pubmed_papers.csv


NameError: name '_combine_text' is not defined

## Comparison: Seed Papers vs. PubMed Results

> Find papers in the PubMed pull that are **furthest from the seed set** (highest dissimilarity).
> Extract key terms from those outliers to identify new exclusion criteria. This uses the same
> SPECTER embeddings from earlier to score semantic similarity.

In [35]:
# Load the seed papers and pubmed results
df_seed_compare = pd.read_csv("seed_papers.csv", encoding="latin-1")
df_seed_compare["_text"] = df_seed_compare.apply(_combine_text, axis=1)
df_seed_compare = df_seed_compare[df_seed_compare["_text"].str.strip().astype(bool)].reset_index(drop=True)

df_pubmed_all = pd.read_csv("pubmed_papers.csv", encoding="utf-8")
df_pubmed_all["_text"] = df_pubmed_all.apply(_combine_text, axis=1)
df_pubmed_all = df_pubmed_all[df_pubmed_all["_text"].str.strip().astype(bool)].reset_index(drop=True)

# Filter to papers with actual abstracts (not just titles, NaN, or the literal string "nan")
# This ensures we have meaningful text for comparison
df_pubmed_all = df_pubmed_all[
    (df_pubmed_all["abstract"].notna()) &  # Not actual NaN/None
    (df_pubmed_all["abstract"].astype(str).str.strip() != "") &  # Not empty strings
    (df_pubmed_all["abstract"].astype(str) != "nan")  # Not the literal string "nan"
].reset_index(drop=True)

print(f"Seed papers: {len(df_seed_compare)}")
print(f"PubMed papers with valid abstracts: {len(df_pubmed_all)}")

Seed papers: 55
PubMed papers with valid abstracts: 212


In [36]:
# Diagnostic: Identify NaNs and nulls in pubmed_papers.csv

print("=" * 80)
print("PUBMED PAPERS — NULL/NaN ANALYSIS")
print("=" * 80)

# Reload raw data to see what's actually in the CSV
df_pubmed_raw = pd.read_csv("pubmed_papers.csv", encoding="utf-8")

print(f"\nTotal rows in CSV: {len(df_pubmed_raw)}")
print(f"\nColumn data types:")
print(df_pubmed_raw.dtypes)

print(f"\n\nNull/NaN counts per column:")
print(df_pubmed_raw.isnull().sum())

print(f"\n\nAbstract column analysis:")
print(f"  - Total rows: {len(df_pubmed_raw)}")
print(f"  - NaN/None: {df_pubmed_raw['abstract'].isna().sum()}")
print(f"  - Empty strings: {(df_pubmed_raw['abstract'].astype(str).str.strip() == '').sum()}")
print(f"  - Rows with 'nan' string literal: {(df_pubmed_raw['abstract'].astype(str) == 'nan').sum()}")
print(f"  - Non-empty abstracts: {(df_pubmed_raw['abstract'].astype(str).str.strip() != '').sum()}")

print(f"\n\nSample of 'abstract' column values:")
for i, val in enumerate(df_pubmed_raw['abstract'].head(20)):
    print(f"  Row {i}: {repr(val)[:80]}")

print(f"\n\nAfter filtering to papers with abstracts:")
df_pubmed_filtered = df_pubmed_raw[
    df_pubmed_raw["abstract"].astype(str).str.strip() != ""
].reset_index(drop=True)
print(f"  Rows remaining: {len(df_pubmed_filtered)}")
print(f"  Rows removed: {len(df_pubmed_raw) - len(df_pubmed_filtered)}")

PUBMED PAPERS — NULL/NaN ANALYSIS

Total rows in CSV: 255

Column data types:
PMID                int64
Title                 str
Authors               str
Citation              str
First Author          str
Journal/Book          str
Publication Year    int64
Create Date           str
PMCID                 str
NIHMS ID              str
DOI                   str
abstract              str
dtype: object


Null/NaN counts per column:
PMID                  0
Title                 0
Authors               0
Citation              0
First Author          0
Journal/Book          0
Publication Year      0
Create Date           0
PMCID                81
NIHMS ID            248
DOI                   1
abstract             43
dtype: int64


Abstract column analysis:
  - Total rows: 255
  - NaN/None: 43
  - Empty strings: 0
  - Rows with 'nan' string literal: 0
  - Non-empty abstracts: 255


Sample of 'abstract' column values:
  Row 0: "The question whether life style may impair the advent or course 

In [17]:
# Load model if not already loaded (from earlier SPECTER section)
from sentence_transformers import SentenceTransformer, util

try:
    model
except NameError:
    print("Loading SPECTER model...")
    model = SentenceTransformer("allenai-specter")
    print("Model loaded.")

# Embed seed papers and compute centroid
print("\nEmbedding seed papers...")
seed_texts_compare = df_seed_compare["_text"].tolist()
seed_embeddings_compare = model.encode(seed_texts_compare, show_progress_bar=False, convert_to_tensor=True)
seed_centroid_compare = seed_embeddings_compare.mean(dim=0)

# Embed PubMed papers and score against seed centroid
print("Embedding PubMed papers (may take a moment)...")
pubmed_texts = df_pubmed_all["_text"].tolist()
pubmed_embeddings = model.encode(pubmed_texts, show_progress_bar=True, convert_to_tensor=True)
similarity_scores = util.cos_sim(seed_centroid_compare.unsqueeze(0), pubmed_embeddings)[0].cpu().numpy()

df_pubmed_all["similarity_to_seed"] = similarity_scores
df_pubmed_all = df_pubmed_all.sort_values("similarity_to_seed", ascending=True).reset_index(drop=True)

c:\Users\rchua\sysrev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SPECTER model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6030.30it/s]


Model loaded.

Embedding seed papers...
Embedding PubMed papers (may take a moment)...


Batches: 100%|██████████| 8/8 [00:22<00:00,  2.82s/it]


In [40]:
# Show the outliers (papers most dissimilar to seed set)
OUTLIER_N = 20
print(f"\nBottom {OUTLIER_N} papers (furthest from seed — potential exclusion candidates):")
print("=" * 100)

# First, check what columns are available
print(f"DEBUG: Available columns in df_pubmed_all: {df_pubmed_all.columns.tolist()}")
print(f"DEBUG: Shape of df_pubmed_all: {df_pubmed_all.shape}")
print(f"DEBUG: First few rows:\n{df_pubmed_all.head(3)}\n")

# Display each outlier paper with its details using direct column access instead of .get()
for idx, row in df_pubmed_all.head(OUTLIER_N).iterrows():
    sim = row["similarity_to_seed"] if "similarity_to_seed" in df_pubmed_all.columns else "N/A"
    title = row.get("title", "N/A") if "title" in df_pubmed_all.columns else row.get("Title", "N/A")
    doi = row.get("doi", "N/A") if "doi" in df_pubmed_all.columns else row.get("DOI", "N/A")
    
    # Convert similarity to float if it's numeric
    try:
        sim = float(sim)
        sim_str = f"{sim:.4f}"
    except (ValueError, TypeError):
        sim_str = str(sim)
    
    print(f"\n{idx + 1}. Similarity: {sim_str} | DOI: {doi}")
    print(f"   Title: {title}")

print("\n" + "=" * 100)

# Extract terms from the outlier papers
outlier_texts = df_pubmed_all.head(OUTLIER_N)["_text"].tolist()
outlier_combined = " ".join(outlier_texts)


Bottom 20 papers (furthest from seed — potential exclusion candidates):
DEBUG: Available columns in df_pubmed_all: ['PMID', 'Title', 'Authors', 'Citation', 'First Author', 'Journal/Book', 'Publication Year', 'Create Date', 'PMCID', 'NIHMS ID', 'DOI', 'abstract', '_text']
DEBUG: Shape of df_pubmed_all: (212, 13)
DEBUG: First few rows:
       PMID  \
0  35606622   
1  40338549   
2  38115046   

                                                                                                 Title  \
0                                                                   Life style and Parkinson's disease   
1                                              Proximity to Golf Courses and Risk of Parkinson Disease   
2  Untargeted serum metabolomics reveals novel metabolite associations and disruptions in amino aci...   

                                                                                               Authors  \
0                        Reichmann H, Csoti I, Koschel J, Lorenzl S, Sc

In [41]:
# Vectorize outlier papers to find their top terms
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Note: max_df is set to 1.0 since we're vectorizing a single combined document
outlier_vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", max_df=1.0)
X_outliers = outlier_vectorizer.fit_transform([outlier_combined])
outlier_tfidf = np.asarray(X_outliers.sum(axis=0)).ravel()
outlier_names = outlier_vectorizer.get_feature_names_out()

outlier_keywords = pd.DataFrame({
    "term": outlier_names,
    "tfidf_score": outlier_tfidf
}).sort_values("tfidf_score", ascending=False)

In [46]:
# Compare against existing search strategy to find new exclusion candidates
from search_strategy import EXCLUSION_TERMS, INCLUSION_CRITERIA, ALTERNATE_TERMS
import re

# Define normalization function if not already available
try:
    _normalize_term("test")
except NameError:
    def _normalize_term(s):
        s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
        return re.sub(r"\s+", " ", s).strip()

# Build sets of all terms already in the strategy (inclusion, alternate, and exclusion)
existing_inclusions = set()
for group in list(INCLUSION_CRITERIA.values()) + list(ALTERNATE_TERMS.values()):
    for t in group:
        existing_inclusions.add(_normalize_term(t.replace("*", "")))

existing_exclusions = {_normalize_term(t.replace("*", "")) for t in EXCLUSION_TERMS}
all_strategy_terms = existing_inclusions | existing_exclusions

outlier_keywords["term_norm"] = outlier_keywords["term"].apply(_normalize_term)
outlier_keywords["in_inclusion"] = outlier_keywords["term_norm"].isin(existing_inclusions)
outlier_keywords["in_exclusion"] = outlier_keywords["term_norm"].isin(existing_exclusions)
outlier_keywords["already_in_strategy"] = outlier_keywords["term_norm"].isin(all_strategy_terms)

# New exclusion candidates: terms NOT in any part of current strategy
OUTLIER_TERMS_N = 50  # or however many you want
new_exclusion_candidates = outlier_keywords[~outlier_keywords["already_in_strategy"]].head(OUTLIER_TERMS_N)

print(f"\nTop {OUTLIER_TERMS_N} terms from outlier papers NOT already in search strategy:")
print("(Showing only terms that are not in INCLUSION_CRITERIA, ALTERNATE_TERMS, or EXCLUSION_TERMS)\n")
display(new_exclusion_candidates[["term", "tfidf_score", "in_inclusion", "in_exclusion"]].style.format({"tfidf_score": "{:.3f}"}))

print("\n💡 Next steps:")
print(f"  1. Review the bottom {OUTLIER_N} papers above — spot-check titles to identify thematic noise.")
print(f"  2. From the terms above, select high-scoring ones that represent true noise/irrelevance.")
print(f"  3. Add those terms to EXCLUSION_TERMS in search_strategy.py.")
print(f"  4. Re-run the PubMed search to measure the precision improvement.")


Top 50 terms from outlier papers NOT already in search strategy:
(Showing only terms that are not in INCLUSION_CRITERIA, ALTERNATE_TERMS, or EXCLUSION_TERMS)



,term,tfidf_score,in_inclusion,in_exclusion
2371,pd,0.557,False,False
2843,risk,0.205,False,False
1065,disease,0.179,False,False
495,associated,0.158,False,False
3086,studies,0.147,False,False
307,95,0.142,False,False
2311,parkinson disease,0.142,False,False
719,ci,0.126,False,False
309,95 ci,0.116,False,False
3112,study,0.105,False,False



💡 Next steps:
  1. Review the bottom 20 papers above — spot-check titles to identify thematic noise.
  2. From the terms above, select high-scoring ones that represent true noise/irrelevance.
  3. Add those terms to EXCLUSION_TERMS in search_strategy.py.
  4. Re-run the PubMed search to measure the precision improvement.


In [43]:
# Alternative approach: Show outlier papers directly + simple word frequency analysis

print("=" * 80)
print("OUTLIER PAPERS — Direct inspection of titles and abstracts")
print("=" * 80)
print(f"\nBottom {OUTLIER_N} papers (lowest similarity to seed):\n")

for idx, row in df_pubmed_all.head(OUTLIER_N).iterrows():
    sim = row.get("similarity_to_seed", "N/A")
    title = row.get("title", "N/A")
    abstract = row.get("abstract", "N/A")
    if isinstance(abstract, str) and len(abstract) > 200:
        abstract = abstract[:200] + "..."
    
    print(f"{idx + 1}. [{sim:.3f}] {title}")
    print(f"   {abstract}\n")

# Simple word frequency from outlier papers (ignore stopwords)
from collections import Counter
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
stop = set(stopwords.words('english'))

# Tokenize outlier texts, filter stopwords and short terms
all_words = []
for text in outlier_texts:
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())  # 3+ char words only
    all_words.extend([w for w in words if w not in stop])

word_freq = Counter(all_words)
top_words = word_freq.most_common(50)

print("=" * 80)
print("WORD FREQUENCY IN OUTLIER PAPERS (Top 50)")
print("=" * 80)
print(f"Total unique words: {len(word_freq)}\n")
for i, (word, count) in enumerate(top_words, 1):
    print(f"  {i:2d}. {word:20s} {count:3d} times")


OUTLIER PAPERS — Direct inspection of titles and abstracts

Bottom 20 papers (lowest similarity to seed):



ValueError: Unknown format code 'f' for object of type 'str'

In [49]:
# Fetch full papers from PubMed for SPECTER results

# Load the SPECTER-ranked CSV
df_specter_results = pd.read_csv("C:/Users/rchua/sysrev/csv-parkinsonT-set_SPECTER.csv")
print(f"Loaded {len(df_specter_results)} papers from SPECTER results")

# Detect PMID column
possible_pmid_cols = ["pubmed_id", "PMID", "pmid", "PubmedID"]
pmid_col_specter = None

for col in possible_pmid_cols:
    if col in df_specter_results.columns:
        pmid_col_specter = col
        print(f"Found PMID column: '{pmid_col_specter}'")
        break

if not pmid_col_specter:
    pmid_cols = [c for c in df_specter_results.columns if "pmid" in c.lower()]
    if not pmid_cols:
        pmid_cols = [c for c in df_specter_results.columns if c.lower() == "id"]
    if pmid_cols:
        pmid_col_specter = pmid_cols[0]
        print(f"Found PMID column: '{pmid_col_specter}'")
    else:
        print("Available columns:")
        for c in df_specter_results.columns:
            print(f"  - {c}")
        raise ValueError("No PMID column detected in SPECTER results.")

# Function to fetch full paper data from PubMed
def _fetch_full_paper(pmid):
    """Fetch full paper data from PubMed using PMID."""
    try:
        results = list(pubmed.query(f"{pmid}[PMID]", max_results=1))
        if results:
            paper = results[0]
            
            # Handle authors - they can be objects or dicts
            authors_str = ""
            if hasattr(paper, 'authors') and paper.authors:
                author_names = []
                for author in paper.authors:
                    if isinstance(author, dict):
                        # Dictionary format
                        name = author.get('name', '') or author.get('lastname', '')
                        if name:
                            author_names.append(name)
                    else:
                        # Object format
                        try:
                            name = f"{author.lastname} {author.initials}"
                            author_names.append(name)
                        except:
                            author_names.append(str(author))
                authors_str = ", ".join(author_names)
            
            return {
                "pmid": int(pmid),  # Ensure PMID is int to match original column type
                "title": paper.title if hasattr(paper, 'title') else "",
                "abstract": paper.abstract if hasattr(paper, 'abstract') else "",
                "authors": authors_str,
                "publication_date": str(paper.publication_date) if hasattr(paper, 'publication_date') else "",
                "journal": paper.journal if hasattr(paper, 'journal') else "",
                "doi": paper.doi if hasattr(paper, 'doi') else "",
                "keywords": ", ".join(paper.keywords) if hasattr(paper, 'keywords') and paper.keywords else "",
            }
        return {"pmid": int(pmid), "title": "", "abstract": "", "authors": "", "publication_date": "", "journal": "", "doi": "", "keywords": ""}
    except Exception as e:
        print(f"  Error fetching PMID {pmid}: {e}")
        return {"pmid": int(pmid), "title": "", "abstract": "", "authors": "", "publication_date": "", "journal": "", "doi": "", "keywords": ""}
    finally:
        time.sleep(0.3)

print(f"\nFetching full papers from PubMed (this may take a few minutes)...")
full_papers_list = []
for i, pmid in enumerate(df_specter_results[pmid_col_specter], 1):
    if i % 10 == 0:
        print(f"  Fetched {i}/{len(df_specter_results)} papers...")
    paper_data = _fetch_full_paper(pmid)
    full_papers_list.append(paper_data)

df_full_papers = pd.DataFrame(full_papers_list)

# Ensure PMID columns are the same type (int) before merging
df_specter_results[pmid_col_specter] = df_specter_results[pmid_col_specter].astype(int)
df_full_papers["pmid"] = df_full_papers["pmid"].astype(int)

# Merge with original SPECTER data
df_specter_full = df_specter_results.merge(df_full_papers, left_on=pmid_col_specter, right_on="pmid", how="left")

# Save to CSV
output_filename = "csv-parkinsonT-set_SPECTER_FULL.csv"
df_specter_full.to_csv(output_filename, index=False, encoding="utf-8")
print(f"\n✓ Exported {len(df_specter_full)} full papers to {output_filename}")
print(f"  File location: {Path(output_filename).resolve()}")
print(f"\nColumns in output: {df_specter_full.columns.tolist()}")

Loaded 197 papers from SPECTER results
Found PMID column: 'PMID'

Fetching full papers from PubMed (this may take a few minutes)...
  Fetched 10/197 papers...
  Fetched 20/197 papers...
  Fetched 30/197 papers...
  Fetched 40/197 papers...
  Fetched 50/197 papers...
  Fetched 60/197 papers...
  Fetched 70/197 papers...
  Fetched 80/197 papers...
  Fetched 90/197 papers...
  Fetched 100/197 papers...
  Fetched 110/197 papers...
  Fetched 120/197 papers...
  Fetched 130/197 papers...
  Fetched 140/197 papers...
  Fetched 150/197 papers...
  Fetched 160/197 papers...
  Fetched 170/197 papers...
  Fetched 180/197 papers...
  Fetched 190/197 papers...

✓ Exported 197 full papers to csv-parkinsonT-set_SPECTER_FULL.csv
  File location: C:\Users\rchua\sysrev\csv-parkinsonT-set_SPECTER_FULL.csv

Columns in output: ['PMID', 'Title', 'Authors', 'Citation', 'First Author', 'Journal/Book', 'Publication Year', 'Create Date', 'PMCID', 'NIHMS ID', 'DOI', 'pmid', 'title', 'abstract', 'authors', 'public